# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row represents one unique content item per client (`client_id` + `content_id`) evaluated at an aggregated 90-day trailing window level using the warehouse release dataset.

**Time Window:** Evaluated across the trailing panel window from 2025-01-27 to 2026-06-30, focusing specifically on a partitioned mid-panel reference month (`month=2026-03`) for initial feature validation and stability checks[cite: 3].

In [4]:
import getpass, os
from huggingface_hub import hf_hub_download
import duckdb

token = getpass.getpass("Enter your Hugging Face READ token: ")
token = token.strip()
os.environ["HF_TOKEN"] = token

# ---- Download the file locally (huggingface_hub handles Xet properly, with retries) ----
local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=token,
)
print("Downloaded to:", local_path)

# ---- Now query it locally, no network flakiness involved ----
con = duckdb.connect()

query = f"""
SELECT client_hash_id, content_hash_id, COUNT(*) as c
FROM read_parquet('{local_path}')
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
GROUP BY client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
"""

result = con.execute(query).fetchall()
print("\nQuery result:")
for row in result:
    print(row)

Enter your Hugging Face READ token: ··········


fact_content_daily_performance/month=202(…):   0%|          | 0.00/124M [00:00<?, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet

Query result:
('client_73cda7b4e4f265ea', 'content_7a105f548d9c6916', 31)
('client_73cda7b4e4f265ea', 'content_a3ea9792f793ec72', 31)
('client_73cda7b4e4f265ea', 'content_36c36abc7650d7af', 31)
('client_73cda7b4e4f265ea', 'content_a7da352b73b02668', 31)
('client_73cda7b4e4f265ea', 'content_f39be42b42a4e8f6', 31)


In [7]:
query = f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as c
FROM read_parquet('{local_path}')
GROUP BY client_hash_id, content_hash_id, report_date
HAVING c > 1
ORDER BY c DESC
LIMIT 10
"""

df = con.execute(query).df()

print("=" * 60)
print("TRUE DUPLICATE CHECK (same client + content + date)")
print("=" * 60)

if df.empty:
    print("✅ No true duplicates found in this dataset.\n")

    # Show a presentable summary instead of a blank output
    summary = con.execute(f"""
        SELECT COUNT(*) AS total_rows,
               COUNT(DISTINCT client_hash_id) AS unique_clients,
               COUNT(DISTINCT content_hash_id) AS unique_contents,
               MIN(report_date) AS earliest_date,
               MAX(report_date) AS latest_date
        FROM read_parquet('{local_path}')
    """).df()

    print("📊 Dataset Summary:")
    print(summary.to_string(index=False))

else:
    print(f"⚠️ Found {len(df)} duplicate group(s):\n")
    print(df.to_string(index=False))

print("=" * 60)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

TRUE DUPLICATE CHECK (same client + content + date)
✅ No true duplicates found in this dataset.

📊 Dataset Summary:
 total_rows  unique_clients  unique_contents earliest_date latest_date
    9841378              55           331437    2026-03-01  2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

* **Feature:** `gsc_avg_position`, `gsc_impressions`, `gsc_clicks`, `ga4_pageviews`, `content_age_days` (knowable prior to the evaluation timestamp).
* **Label / Proxy:** Future traffic delta or observed trend direction over subsequent trailing/leading windows.
* **Context:** `client_id`, `content_id`, `report_date` (used strictly for grain grouping, partitioning, and joins; never learned by the model)[cite: 3].
* **Excluded:** Internal administrative flags or future outcome leakage metrics that reveal post-decision values.

In [14]:
import getpass, os
from huggingface_hub import hf_hub_download
import duckdb

token = getpass.getpass("Enter your Hugging Face READ token: ").strip()
os.environ["HF_TOKEN"] = token

local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=token,
)
print("Downloaded to:", local_path)

con = duckdb.connect()

query_stats = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) as missing_position_pct
FROM read_parquet('{local_path}')
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""

stats_df = con.execute(query_stats).df()
print("📊 March 2026 Content Performance Stats:")
print(stats_df.to_string(index=False))

Enter your Hugging Face READ token: ··········
Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
📊 March 2026 Content Performance Stats:
 total_rows   min_date   max_date  missing_position_pct
    9841378 2026-03-01 2026-03-31              0.633074


In [15]:
from huggingface_hub import snapshot_download

local_dir = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    allow_patterns=["fact_content_daily_performance/month=2026-03/*.parquet"],
    token=token,
)

query_stats = f"""
SELECT
    COUNT(*) as total_rows,
    MIN(report_date) as min_date,
    MAX(report_date) as max_date,
    AVG(CASE WHEN gsc_avg_position IS NULL THEN 1.0 ELSE 0 END) as missing_position_pct
FROM read_parquet('{local_dir}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""

stats_df = con.execute(query_stats).df()
print("📊 March 2026 Content Performance Stats:")
print(stats_df.to_string(index=False))

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

📊 March 2026 Content Performance Stats:
 total_rows   min_date   max_date  missing_position_pct
    9841378 2026-03-01 2026-03-31              0.633074


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The queries above empirically validate that our row grain holds cleanly with zero overlapping duplicates, and show explicit date boundaries and null ratios across the targeted monthly partition[cite: 3, 6].

In [16]:
import getpass, os
from huggingface_hub import hf_hub_download
import duckdb

token = getpass.getpass("Enter your Hugging Face READ token: ").strip()
os.environ["HF_TOKEN"] = token

local_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=token,
)
print("Downloaded to:", local_path)

con = duckdb.connect()

query_window = f"""
SELECT
    client_hash_id,
    MIN(report_date) as client_min_date,
    MAX(report_date) as client_max_date,
    COUNT(*) as row_count
FROM read_parquet('{local_path}')
GROUP BY client_hash_id
ORDER BY row_count DESC
LIMIT 5
"""

window_df = con.execute(query_window).df()
print("📅 Client Date Range & Row Counts:")
print(window_df.to_string(index=False))

Enter your Hugging Face READ token: ··········
Downloaded to: /root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
📅 Client Date Range & Row Counts:
         client_hash_id client_min_date client_max_date  row_count
client_625b6439094e23e4      2026-03-01      2026-03-31     988497
client_3ffa76342f366962      2026-03-01      2026-03-31     904847
client_73cda7b4e4f265ea      2026-03-01      2026-03-31     869640
client_08a6a72ff48e62c0      2026-03-01      2026-03-31     851275
client_62f4a7e64f5e0096      2026-03-01      2026-03-31     756660


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset provides observed, measured, and directional operational indicators[cite: 5]. It cannot offer causal proof (e.g., verifying that a specific editorial rewrite directly generated a rank shift) or account for historical recording discrepancies prior to individual client integration boundaries[cite: 3].

In [17]:
query_leakage_test = f"""
SELECT
    COUNT(*) as evaluated_rows
FROM read_parquet('{local_path}')
WHERE ga4_data_available IS TRUE
  AND report_date >= '2026-03-01' AND report_date <= '2026-03-31'
"""

leakage_df = con.execute(query_leakage_test).df()
print("🔍 GA4 Data Leakage Check (March 2026):")
print(leakage_df.to_string(index=False))

🔍 GA4 Data Leakage Check (March 2026):
 evaluated_rows
         413966


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.